# rsc GPU — FULL Workflow (with regress_out + scale)

Matches the official rsc paper (Dicks et al. 2026, Figure 2a).

Change `DATASET` below to switch datasets.

In [1]:
# === CHANGE THIS TO SWITCH DATASETS ===
DATASET = "mouse_brain_1m"   # or "lung65k" or "mouse_brain_1m"

import json, time, os, inspect
import numpy as np
import pandas as pd
import cupy as cp
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator

rmm.reinitialize(pool_allocator=True, managed_memory=True)
cp.cuda.set_allocator(rmm_cupy_allocator)

import rapids_singlecell as rsc
import scanpy as sc

sc.settings.verbosity = 0
print(f"rsc: {rsc.__version__}")
print(f"Scanpy: {sc.__version__}")
os.makedirs("write", exist_ok=True)

rsc: 0.14.1
Scanpy: 1.12


/tmp/ipykernel_1268863/1201081642.py:19: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f"Scanpy: {sc.__version__}")


In [2]:
with open("benchmark_config.json") as f:
    cfg = json.load(f)
gcfg = cfg["global"]
dcfg = cfg["datasets"][DATASET]
prefix = dcfg["pipeline_prefix"]
timings = {}

out = f"write/{prefix}_rsc_gpu_0141_full"
print(f"Dataset: {dcfg['name']}")
print(f"Output prefix: {out}")

Dataset: Mouse Brain 1.3M
Output prefix: write/mouse_1m_rsc_gpu_0141_full


## Load + GPU transfer

For mouse_brain_1m: auto-subsets to 1M cells (int32 sparse ceiling).

In [3]:
t0 = time.time()
adata = sc.read_h5ad(dcfg["canonical_h5ad"])
timings["load"] = time.time() - t0
print(f"Loaded in {timings['load']:.1f}s: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
assert "counts" in adata.layers

# Subset to 1M for mouse brain
N_CELLS = 1_000_000
full_n_obs = adata.n_obs
if adata.n_obs > N_CELLS:
    print(f"Subsetting to {N_CELLS:,} cells (from {adata.n_obs:,})")
    adata = adata[:N_CELLS, :].copy()

t0 = time.time()
rsc.get.anndata_to_GPU(adata)
timings["to_gpu"] = time.time() - t0
print(f"Moved to GPU in {timings['to_gpu']:.1f}s")

Loaded in 147.8s: 1,153,539 cells x 22,788 genes
Subsetting to 1,000,000 cells (from 1,153,539)
Moved to GPU in 5.6s


## Normalize + log1p

In [4]:
t0 = time.time()
rsc.pp.normalize_total(adata, target_sum=gcfg["target_sum"])
rsc.pp.log1p(adata)
cp.cuda.Device(0).synchronize()
timings["normalize_log1p"] = time.time() - t0
print(f"Normalize + log1p: {timings['normalize_log1p']:.1f}s")

Normalize + log1p: 2.2s


## HVG

In [5]:
import cupyx.scipy.sparse as cusp
from scipy import sparse as sp

if sp.issparse(adata.layers["counts"]):
    print("Converting counts layer to GPU...")
    adata.layers["counts"] = cusp.csr_matrix(adata.layers["counts"])

t0 = time.time()
rsc.pp.highly_variable_genes(
    adata, layer="counts", n_top_genes=dcfg["n_top_genes"],
    flavor="seurat_v3",
)
timings["hvg"] = time.time() - t0
print(f"HVG ({adata.var['highly_variable'].sum()} genes): {timings['hvg']:.1f}s")

Converting counts layer to GPU...
HVG (5000 genes): 7.1s


## Subset to HVGs + Regress out + Scale

**This is what differs from the minimal workflow.**

Matches the rsc 1M brain demo notebook:
```python
rsc.pp.regress_out(adata, keys=["total_counts", "pct_counts_MT"])
rsc.pp.scale(adata, max_value=10)
```

In [6]:
adata = adata[:, adata.var["highly_variable"]].copy()
print(f"Subset to HVGs: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

Subset to HVGs: 1,000,000 cells x 5,000 genes


In [7]:
# Auto-detect MT column name
mt_col = None
for candidate in ["pct_counts_mt", "pct_counts_MT", "pct_counts_Mt"]:
    if candidate in adata.obs.columns:
        mt_col = candidate
        break

regress_keys = ["total_counts"]
if mt_col:
    regress_keys.append(mt_col)
print(f"Regress out keys: {regress_keys}")

t0 = time.time()
rsc.pp.regress_out(adata, keys=regress_keys)
timings["regress_out"] = time.time() - t0
print(f"Regress out: {timings['regress_out']:.1f}s")

Regress out keys: ['total_counts', 'pct_counts_mt']
Regress out: 1.9s


In [8]:
t0 = time.time()
rsc.pp.scale(adata, max_value=10)
timings["scale"] = time.time() - t0
print(f"Scale: {timings['scale']:.1f}s")

Scale: 1.9s


## PCA (on scaled data)

In [9]:
t0 = time.time()
rsc.pp.pca(adata, n_comps=gcfg["pca_n_comps"])
timings["pca"] = time.time() - t0
print(f"PCA: {timings['pca']:.1f}s")

PCA: 11.9s


## Neighbors

In [10]:
t0 = time.time()
rsc.pp.neighbors(
    adata, n_neighbors=dcfg["n_neighbors"], n_pcs=dcfg["neighbors_n_pcs"],
    use_rep="X_pca", algorithm="brute",
    metric=gcfg["neighbor_metric"], random_state=gcfg["random_state"],
)
timings["neighbors"] = time.time() - t0
print(f"Neighbors: {timings['neighbors']:.1f}s")

Neighbors: 12.5s


## Leiden

In [11]:
t0 = time.time()
leiden_kwargs = {
    "resolution": dcfg["leiden_resolution"],
    "random_state": gcfg["random_state"],
    "key_added": "leiden",
}
if gcfg.get("rsc_leiden_n_iterations") is not None:
    leiden_kwargs["n_iterations"] = gcfg["rsc_leiden_n_iterations"]
rsc.tl.leiden(adata, **leiden_kwargs)
timings["leiden"] = time.time() - t0
n_clusters = adata.obs["leiden"].nunique()
print(f"Leiden ({n_clusters} clusters): {timings['leiden']:.1f}s")

Leiden (36 clusters): 3.0s


## Checkpoint: save clusters

In [12]:
rsc.get.anndata_to_CPU(adata)
pd.DataFrame({
    "barcode": adata.obs_names.astype(str),
    "leiden": adata.obs["leiden"].astype(str).values,
}).to_csv(f"{out}_clusters.csv", index=False)
rsc.get.anndata_to_GPU(adata)
print(f"Checkpoint: clusters saved ({n_clusters} clusters)")

Checkpoint: clusters saved (36 clusters)


## UMAP

In [13]:
t0 = time.time()
rsc.tl.umap(
    adata, min_dist=gcfg["umap_min_dist"], spread=gcfg["umap_spread"],
    init_pos="spectral", random_state=gcfg["random_state"],
)
timings["umap"] = time.time() - t0
print(f"UMAP: {timings['umap']:.1f}s")

UMAP: 2.0s


## Differential Expression

In [14]:
t0 = time.time()
rank_sig = inspect.signature(rsc.tl.rank_genes_groups)
rank_kwargs = {
    "groupby": "leiden",
    "method": gcfg["de_method"],
    "corr_method": gcfg["de_corr_method"],
    "use_raw": False,
    "pts": True,
}
for key, value in {"pre_load": True, "tie_correct": False, "use_continuity": False}.items():
    if key in rank_sig.parameters:
        rank_kwargs[key] = value
rsc.tl.rank_genes_groups(adata, **rank_kwargs)
timings["de"] = time.time() - t0
print(f"DE: {timings['de']:.1f}s")

DE: 4.3s


/home/sjeong7/.local/lib/python3.13/site-packages/rapids_singlecell/tools/_rank_genes_groups/_core.py:485: RuntimeWarning: invalid value encountered in log2
  stats_data[group_name, "logfoldchanges"] = np.log2(


## Move back to CPU + Save all outputs

In [15]:
t0 = time.time()
rsc.get.anndata_to_CPU(adata)
timings["to_cpu"] = time.time() - t0

# Clusters (overwrite checkpoint)
pd.DataFrame({
    "barcode": adata.obs_names.astype(str),
    "leiden": adata.obs["leiden"].astype(str).values,
}).to_csv(f"{out}_clusters.csv", index=False)

# Markers
markers = sc.get.rank_genes_groups_df(adata, group=None)
markers.to_csv(f"{out}_markers.csv", index=False)
markers_filt = markers[(markers["pvals_adj"] < 0.05) & (markers["logfoldchanges"] > 0.1)]
markers_filt.to_csv(f"{out}_markers_filtered.csv", index=False)

# UMAP
pd.DataFrame(
    adata.obsm["X_umap"], index=adata.obs_names.astype(str),
    columns=["UMAP_1", "UMAP_2"],
).reset_index(names="barcode").to_csv(f"{out}_umap.csv", index=False)

# h5ad
t0s = time.time()
adata.write_h5ad(f"{out}.h5ad", compression="gzip")
timings["save_h5ad"] = time.time() - t0s

# Spec
spec = {
    "pipeline": "rsc_gpu_0141_full",
    "dataset": dcfg["name"],
    "workflow": "FULL (with regress_out + scale)",
    "rapids_singlecell_version": rsc.__version__,
    "regress_out_keys": regress_keys,
    "scale_max_value": 10,
    "cell_subset": {
        "n_cells_used": int(adata.n_obs),
        "n_cells_total": full_n_obs,
    },
    "timings_seconds": timings,
    "results": {
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
        "n_hvg": dcfg["n_top_genes"],
        "n_clusters": n_clusters,
        "n_marker_rows": len(markers),
        "n_marker_rows_filtered": len(markers_filt),
    },
}
with open(f"{out}_spec.json", "w") as f:
    json.dump(spec, f, indent=2)

print("\n=== Timings ===")
total = 0
for step, t in timings.items():
    print(f"  {step:20s}: {t:8.1f}s")
    total += t
print(f"  {'TOTAL':20s}: {total:8.1f}s ({total/60:.1f} min)")
print(f"\nCells: {adata.n_obs:,} (of {full_n_obs:,} total)")
print(f"Clusters: {n_clusters}")
print(f"Filtered markers: {len(markers_filt)}")


=== Timings ===
  load                :    147.8s
  to_gpu              :      5.6s
  normalize_log1p     :      2.2s
  hvg                 :      7.1s
  regress_out         :      1.9s
  scale               :      1.9s
  pca                 :     11.9s
  neighbors           :     12.5s
  leiden              :      3.0s
  umap                :      2.0s
  de                  :      4.3s
  to_cpu              :     10.4s
  save_h5ad           :    404.1s
  TOTAL               :    614.8s (10.2 min)

Cells: 1,000,000 (of 1,153,539 total)
Clusters: 36
Filtered markers: 61557
